# 00 — Byte-Pair Encoding from First Principles

Every KAMUI experiment starts with tokenisation. This notebook trains a
byte-level BPE tokenizer from scratch and inspects every step: the byte
fallback, the learned merges, and the lossless round-trip.

**Key idea**: BPE starts from 256 byte tokens (so *any* string is
representable) and greedily merges the most frequent adjacent pair until
the vocabulary budget is reached.

In [ ]:
import sys
sys.path.insert(0, "..")  # run from the notebooks/ directory

import matplotlib
import torch

import kamui
from kamui.model.config import ModelConfig

torch.manual_seed(0)
print("KAMUI", kamui.__version__)

In [ ]:
from kamui.tokenizer.bpe import BPETokenizer
from kamui.tokenizer.utils import get_stats, merge_pair, text_to_bytes

# Step 1: text is just bytes.
print(text_to_bytes("hello"))

In [ ]:
# Step 2: count adjacent pairs and apply one merge — the heart of BPE.
sequence = text_to_bytes("aa aab aab")
stats = get_stats([sequence])
best = max(stats, key=stats.get)
print("most frequent pair:", best, "count:", stats[best])
print("after merging it ->", merge_pair(sequence, best, 256))

In [ ]:
# Step 3: train a full tokenizer and inspect what it learned.
corpus = "the cat sat on the mat. the dog sat on the log. " * 50
tokenizer = BPETokenizer.train(corpus, vocab_size=300)
print(tokenizer)
print("encode('the cat') ->", tokenizer.encode("the cat"))

In [ ]:
# Step 4: the round-trip is lossless for ANY string — even emoji.
for text in ["the cat sat", "caf\u00e9 \U0001f916", "<|endoftext|>done"]:
    assert tokenizer.decode(tokenizer.encode(text)) == text
print("round-trip lossless \u2713")

**Next**: `01_attention_mechanics` — what the model does with these IDs.